In [4]:
import numpy as np
import pandas as pd
import scipy
import anndata as ad
# import scanpy as sc

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("SciPy:", scipy.__version__)
# print("Scanpy:", sc.__version__)
print("All imports successful!")

NumPy: 2.5.4
Pandas: 3.1.0
SciPy: 1.18.1
All imports successful!


In [5]:
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

In [6]:
from abc_atlas_access.abc_atlas_cache.abc_project_cache import AbcProjectCache

In [7]:
download_base = Path('../../data/abc_atlas')
abc_cache = AbcProjectCache.from_cache_dir(download_base)

abc_cache.current_manifest

c:\Users\lilyg\miniforge3\envs\neuroscience490\Lib\site-packages\abc_atlas_access\abc_atlas_cache\cloud_cache.py:679: OutdatedManifestWarning: You are loading releases/20261031/manifest.json. A more up to date version of the dataset -- releases/20261130/manifest.json -- exists online. To see the changes between the two versions of the dataset, run
type.compare_manifests('releases/20261031/manifest.json', 'releases/20261130/manifest.json')
To load another version of the dataset, run
type.load_manifest('releases/20261130/manifest.json')
  warnings.warn(msg, OutdatedManifestWarning)


'releases/20261031/manifest.json'

In [8]:
abc_cache.list_metadata_files('WHB-10Xv3')

['anatomical_division_structure_map',
 'cell_metadata',
 'donor',
 'example_genes_all_cells_expression',
 'gene',
 'region_of_interest_structure_map']

In [9]:
abc_cache.list_metadata_files('WHB-taxonomy')

['cluster',
 'cluster_annotation_term',
 'cluster_annotation_term_set',
 'cluster_to_cluster_annotation_membership']

In [10]:
cell = abc_cache.get_metadata_dataframe(
    directory='WHB-10Xv3',
    file_name='cell_metadata',
    dtype={'cell_label': str}
)
cell.set_index('cell_label', inplace=True)
print("Number of cells = ", len(cell))
cell.head(5)
cell.columns

Number of cells =  3369219


C:\Users\lilyg\AppData\Local\Temp\ipykernel_12444\1535995828.py:6: Pandas4Warning: The inplace keyword in DataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  cell.set_index('cell_label', inplace=True)


Index(['cell_barcode', 'barcoded_cell_sample_label', 'library_label',
       'feature_matrix_label', 'entity', 'brain_section_label',
       'library_method', 'donor_label', 'donor_sex', 'dataset_label', 'x', 'y',
       'cluster_alias', 'region_of_interest_label',
       'anatomical_division_label', 'abc_sample_id'],
      dtype='str')

In [11]:
region_counts = (
    cell.groupby("region_of_interest_label")
    .agg(
        n_cells=("donor_label", "size"),
        n_donors=("donor_label", "nunique"),
    )
    .sort_values(["n_donors", "n_cells"], ascending=False)
)

print(region_counts)

                          n_cells  n_donors
region_of_interest_label                   
Human M1C                  116576         4
Human MTG                  107301         3
Human CBV                   71874         3
Human SN                    59505         3
Human CA4C-DGC              56367         3
...                           ...       ...
Human HTHma                 10099         1
Human VA                     8573         1
Human CA3R                   8077         1
Human VPL                    6877         1
Human RN                     4714         1

[109 rows x 2 columns]


In [12]:
roi = abc_cache.get_metadata_dataframe(
    directory="WHB-10Xv3",
    file_name="region_of_interest_structure_map",
)

region_names = (
    roi.dropna(subset=["structure_name"])
    .groupby("region_of_interest_label")["structure_name"]
    .agg(lambda names: " / ".join(names.unique()))
)

region_summary = region_counts.join(region_names, validate="one_to_one")
display(region_summary)


,n_cells,n_donors,structure_name
region_of_interest_label,,,
Human M1C,116576,4,"primary motor cortex (area M1, area 4)"
Human MTG,107301,3,middle temporal gyrus
Human CBV,71874,3,cerebellar vermis
Human SN,59505,3,substantia nigra
Human CA4C-DGC,56367,3,caudal dentate gyrus / pyramidal cells of caud...
...,...,...,...
Human HTHma,10099,1,mammillary region of HTH
Human VA,8573,1,ventral anterior nucleus of thalamus
Human CA3R,8077,1,rostral CA3


In [13]:
cortical_labels = cell.loc[
    cell["anatomical_division_label"].eq("Cerebral cortex"),
    "region_of_interest_label",
].unique()

cortical_names = region_summary.loc[
    region_summary.index.isin(cortical_labels), "structure_name"
]

print("\n".join(cortical_names.dropna().unique()))

primary motor cortex (area M1, area 4)
middle temporal gyrus
medial (posterior) entorhinal cortex
subcentral cortex (gustatory cortex, area 43)
medial division of PPHC (area TH) / lateral division of PPHC (area TL)
superior temporal gyrus
dysgranular insular cortex
rostral division of IPC (area 40)
granular insular cortex
caudal portion of VFC (area 44) / rostral portion of VFC (area 45)
primary somatosensory cortex (area S1, areas 3,1,2)
subgenual (subcallosal) division of MFC (area 25)
temporal polar cortex (area TG, area 38)
piriform cortex
ventral division of PCC (area 23)
rostral division of SPC (area 5) / caudal division of SPC (area 7)
frontal agranular insular cortex (area FI)
caudal division of OFCi (area 13)
area 29 of retrosplenial cortex / area 30 of retrosplenial cortex
primary visual cortex (striate cortex, area V1/17)
primary auditory cortex (core)
lateral (anterior) entorhinal cortex
parastriate cortex (area V2, area 18)
ventral division of MFC (area 24)
caudal division

In [14]:
cortical_names = region_summary.loc[
    region_summary.index.isin(cortical_labels), "structure_name"
]

for label, name in cortical_names.dropna().items():
    print(f"{label} — {name}")

Human M1C — primary motor cortex (area M1, area 4)
Human MTG — middle temporal gyrus
Human MEC — medial (posterior) entorhinal cortex
Human A43 — subcentral cortex (gustatory cortex, area 43)
Human TH-TL — medial division of PPHC (area TH) / lateral division of PPHC (area TL)
Human STG — superior temporal gyrus
Human Idg — dysgranular insular cortex
Human A40 — rostral division of IPC (area 40)
Human Ig — granular insular cortex
Human A44-A45 — caudal portion of VFC (area 44) / rostral portion of VFC (area 45)
Human S1C — primary somatosensory cortex (area S1, areas 3,1,2)
Human A25 — subgenual (subcallosal) division of MFC (area 25)
Human A38 — temporal polar cortex (area TG, area 38)
Human Pir — piriform cortex
Human A23 — ventral division of PCC (area 23)
Human A5-A7 — rostral division of SPC (area 5) / caudal division of SPC (area 7)
Human FI — frontal agranular insular cortex (area FI)
Human A13 — caudal division of OFCi (area 13)
Human A29-A30 — area 29 of retrosplenial cortex / 

In [15]:
mouse_libraries = abc_cache.get_metadata_dataframe(
    directory="Consensus-WMB-Macosko-10X",
    file_name="library",
)

mouse_isocortex_libraries = mouse_libraries.loc[
    mouse_libraries["anatomical_division_label"].eq("Isocortex")
]

display(
    mouse_isocortex_libraries.groupby("region_of_interest_acronym")
    .agg(
        n_libraries=("donor_label", "size"),
        n_donors=("donor_label", "nunique"),
    )
    .sort_index()
)

,n_libraries,n_donors
region_of_interest_acronym,,
ACA,12,2
AUD,12,2
CTX,18,3
MOp,32,12
RSP,15,2
S1,5,1
VIS,6,2
VISp,6,2


In [16]:
mouse_cells = abc_cache.get_metadata_dataframe(
    directory="Consensus-WMB-Macosko-10X",
    file_name="cell_metadata",
    usecols=["cell_label", "library_label", "feature_matrix_label"],
).set_index("cell_label")

mop_libraries = mouse_libraries.loc[
    mouse_libraries["region_of_interest_acronym"].eq("MOp"),
    ["library_label", "region_of_interest_acronym", "donor_label"],
].set_index("library_label")

mouse_mop = mouse_cells.join(
    mop_libraries,
    on="library_label",
    how="inner",
    validate="many_to_one",
)

print(f"MOp cells: {len(mouse_mop):,}")
print(f"Donors with cells: {mouse_mop['donor_label'].nunique()}")
display(mouse_mop.head())

MOp cells: 147,560
Donors with cells: 12


,library_label,feature_matrix_label,region_of_interest_acronym,donor_label
cell_label,,,,
pBICCNsMMrMOpAiF003d190318b_GCCATTCTCTCGCCTA,pBICCNsMMrMOpAiF003d190318b,Macosko-10X-Isocortex,MOp,F003
pBICCNsMMrMOpAiF003d190318b_TCCACCACACTAAACC,pBICCNsMMrMOpAiF003d190318b,Macosko-10X-Isocortex,MOp,F003
pBICCNsMMrMOpAiF003d190318b_CCTCTCCCAAACCATC,pBICCNsMMrMOpAiF003d190318b,Macosko-10X-Isocortex,MOp,F003
pBICCNsMMrMOpAiF003d190318b_CCACCATTCTTACACT,pBICCNsMMrMOpAiF003d190318b,Macosko-10X-Isocortex,MOp,F003
pBICCNsMMrMOpAiF003d190318b_TCCTTCTTCTACCAGA,pBICCNsMMrMOpAiF003d190318b,Macosko-10X-Isocortex,MOp,F003


In [17]:
taxonomy_dir = "Consensus-WMB-integrated-taxonomy"

mouse_cell_clusters = abc_cache.get_metadata_dataframe(
    directory=taxonomy_dir,
    file_name="cell_to_cluster_membership",
    usecols=["cell_label", "cluster_alias"],
).set_index("cell_label")

mouse_membership = abc_cache.get_metadata_dataframe(
    directory=taxonomy_dir,
    file_name="cluster_to_cluster_annotation_membership",
)

mouse_transmitters = (
    mouse_membership.loc[
        mouse_membership["cluster_annotation_term_set_name"].eq("neurotransmitter"),
        ["cluster_alias", "cluster_annotation_term_name"],
    ]
    .drop_duplicates()
    .set_index("cluster_alias")
    .rename(columns={"cluster_annotation_term_name": "neurotransmitter"})
)

mouse_mop_annotated = (
    mouse_mop.join(
        mouse_cell_clusters.reindex(mouse_mop.index),
        validate="one_to_one",
    )
    .join(mouse_transmitters, on="cluster_alias", validate="many_to_one")
)

mouse_counts = (
    mouse_mop_annotated.groupby(["donor_label", "neurotransmitter"], dropna=False)
    .size()
    .unstack(fill_value=0)
)

display(mouse_counts)

neurotransmitter,GABA,Glut,Glut-Chol,Glut-GABA,NN,NaN
donor_label,,,,,,
F003,1570,6308,6,0,4267,8
F004,447,3906,0,0,2645,1
F005,1601,11446,0,3,4723,30
F006,926,8962,0,1,4954,20
F007,859,6145,0,0,3526,5
F008,1766,10654,0,5,6813,11
M002,709,4129,0,0,4163,5
M003,844,5810,0,4,4834,6
M004,726,5361,0,2,3714,12


In [18]:
mouse_pilot = mouse_mop_annotated.loc[
    mouse_mop_annotated["neurotransmitter"].isin(["GABA", "Glut"])
].copy()

mouse_pilot["neuron_group"] = mouse_pilot["neurotransmitter"].map({
    "GABA": "GABAergic",
    "Glut": "Glutamatergic",
})

print(f"Mouse pilot: {len(mouse_pilot):,} cells")
print(f"Donors: {mouse_pilot['donor_label'].nunique()}")
print(mouse_pilot["feature_matrix_label"].value_counts())

Mouse pilot: 94,396 cells
Donors: 12
feature_matrix_label
Macosko-10X-Isocortex    94396
Name: count, dtype: int64


In [19]:
mouse_region_names = {"MOp": "M1", "S1": "S1", "VISp": "V1"}

selected_libraries = mouse_libraries.loc[
    mouse_libraries["region_of_interest_acronym"].isin(mouse_region_names),
    ["library_label", "region_of_interest_acronym", "donor_label"],
].set_index("library_label")

mouse_three_regions = (
    mouse_cells.join(
        selected_libraries, on="library_label", how="inner", validate="many_to_one"
    )
    .join(mouse_cell_clusters, validate="one_to_one")
    .join(mouse_transmitters, on="cluster_alias", validate="many_to_one")
)

mouse_three_regions["region"] = mouse_three_regions[
    "region_of_interest_acronym"
].map(mouse_region_names)

display(
    mouse_three_regions.groupby(["region", "neurotransmitter"], dropna=False)
    .size()
    .unstack(fill_value=0)
)

neurotransmitter,GABA,Glut,Glut-Chol,Glut-GABA,NN,NaN
region,,,,,,
M1,12539,81857,6,23,52974,161
S1,2533,27505,0,0,8908,55
V1,3515,36986,0,3,12944,108


In [21]:
from pathlib import Path

export_dir = Path("../data/region_exports")
export_dir.mkdir(exist_ok=True)

for filename, table in {
    "mouse_M1_S1_V1_metadata.csv": mouse_three_regions
}.items():
    path = export_dir / filename
    table.to_csv(path, index_label="cell_label")
    print(f"{path.resolve()} | {len(table):,} rows | {path.stat().st_size / 1e6:.1f} MB")

C:\Users\lilyg\OneDrive\UPS\Fall 2026 UPS\NRSC 490\human-mouse-cortex\data\region_exports\mouse_M1_S1_V1_metadata.csv | 240,117 rows | 28.8 MB
